# **MLOPS Final Project - Ibrahim's Part**

### **Sections**

* Section 1 — Issue #19 Setup MLflow
* Section 2 — Issue #18 Model evaluation
* Section 3 — Issue #17 Train baseline model
* Section 4 — Issue #20 Log experiments
* Section 5 — Issue #21 Hyperparameter tuning
* Section 6 — Issue #22 Register best model
* Section 7 — Issue #16 DVC pipeline stages

**Best way to Implemenet:**

Setup MLflow → Evaluation → Baseline Training → Logging → HPO → Register Model → DVC

## **Issue #19 — Setup MLflow**

### **Notebook Cell**

In [48]:
# =========================
# Issue #19: Setup MLflow
# File: src/training/mlflow_setup.py
# =========================

from pathlib import Path

Path("src").mkdir(parents=True, exist_ok=True)
Path("src/training").mkdir(parents=True, exist_ok=True)
Path("mlruns").mkdir(parents=True, exist_ok=True)

Path("src/__init__.py").touch()
Path("src/training/__init__.py").touch()

In [49]:
%%writefile src/training/mlflow_setup.py
# =========================
# MLflow setup utilities
# =========================

from pathlib import Path

import mlflow


def configure_mlflow(
    experiment_name="mlops_training_experiments",
    backend_store_path="mlruns/mlflow.db",
):
    """
    Configure MLflow tracking using a local SQLite backend.

    SQLite is used instead of a file-only backend because it supports
    stronger experiment tracking and model registry behavior.
    """
    backend_store_path = Path(backend_store_path)
    backend_store_path.parent.mkdir(parents=True, exist_ok=True)

    tracking_uri = f"sqlite:///{backend_store_path.resolve()}"

    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(experiment_name)

    return tracking_uri

Overwriting src/training/mlflow_setup.py


In [50]:
# =========================
# Verify MLflow setup
# =========================

from src.training.mlflow_setup import configure_mlflow

tracking_uri = configure_mlflow()
print("MLflow tracking URI:", tracking_uri)

MLflow tracking URI: sqlite:////content/mlruns/mlflow.db


In [51]:
# =========================
# Start MLflow UI in background
# =========================

!nohup mlflow ui \
    --backend-store-uri sqlite:///mlruns/mlflow.db \
    --host 0.0.0.0 \
    --port 5000 > mlflow.log 2>&1 &

In [52]:
# =========================
# Open MLflow UI
# =========================

from google.colab import output

output.serve_kernel_port_as_window(5000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

## **Issue #18 — Model evaluation**

### **Notebook Cell**

In [53]:
# =========================
# Issue #18: Model Evaluation
# File: src/evaluation/evaluate.py
# =========================

from pathlib import Path

Path("src/evaluation").mkdir(parents=True, exist_ok=True)
Path("src/evaluation/__init__.py").touch()

In [54]:
%%writefile src/evaluation/evaluate.py
# =========================
# Model evaluation utilities
# =========================

import json
from pathlib import Path

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)


def detect_problem_type(y):
    """
    Detect whether the target is classification or regression.
    """
    y_series = pd.Series(y)
    unique_count = y_series.nunique()

    if y_series.dtype == "object" or unique_count <= 20:
        return "classification"

    return "regression"


def evaluate_model(model, X_test, y_test, output_dir="reports"):
    """
    Evaluate a trained model and save metrics/reports.
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    problem_type = detect_problem_type(y_test)
    y_pred = model.predict(X_test)

    metrics = {
        "problem_type": problem_type,
    }

    if problem_type == "classification":
        metrics["accuracy"] = float(accuracy_score(y_test, y_pred))
        metrics["precision_macro"] = float(
            precision_score(y_test, y_pred, average="macro", zero_division=0)
        )
        metrics["recall_macro"] = float(
            recall_score(y_test, y_pred, average="macro", zero_division=0)
        )
        metrics["f1_macro"] = float(
            f1_score(y_test, y_pred, average="macro", zero_division=0)
        )

        report = classification_report(y_test, y_pred, zero_division=0)
        confusion = confusion_matrix(y_test, y_pred)

        with open(output_path / "classification_report.txt", "w", encoding="utf-8") as file:
            file.write(report)

        pd.DataFrame(confusion).to_csv(output_path / "confusion_matrix.csv", index=False)

    else:
        metrics["mae"] = float(mean_absolute_error(y_test, y_pred))
        metrics["rmse"] = float(mean_squared_error(y_test, y_pred, squared=False))
        metrics["r2"] = float(r2_score(y_test, y_pred))

    with open(output_path / "metrics.json", "w", encoding="utf-8") as file:
        json.dump(metrics, file, indent=4)

    return metrics

Overwriting src/evaluation/evaluate.py


## **Issue #17 — Train baseline model**

### **Branch**

In [55]:
# git checkout develop
# git pull origin develop
# git checkout -b feature/train-baseline-17

### **Notebook Cell**

In [56]:
# =========================
# Issue #17: Train baseline model
# File: src/training/train_baseline.py
# =========================

from pathlib import Path

Path("src/training").mkdir(parents=True, exist_ok=True)
Path("models").mkdir(parents=True, exist_ok=True)
Path("reports").mkdir(parents=True, exist_ok=True)

In [57]:
%%writefile src/training/train_baseline.py
# =========================
# Baseline model training
# =========================

import argparse
import json
from pathlib import Path

import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.evaluation.evaluate import detect_problem_type, evaluate_model


def load_split_data(train_path, test_path, target_column):
    """
    Load train/test split files and separate features from target.
    """
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    if target_column not in train_df.columns:
        raise ValueError(f"Target column '{target_column}' was not found in train data.")

    if target_column not in test_df.columns:
        raise ValueError(f"Target column '{target_column}' was not found in test data.")

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    return X_train, y_train, X_test, y_test


def build_preprocessor(X_train):
    """
    Build preprocessing pipeline for numeric and categorical features.
    """
    numeric_features = X_train.select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

    return preprocessor, numeric_features, categorical_features


def build_baseline_model(problem_type, random_state):
    """
    Build baseline model based on the problem type.
    """
    if problem_type == "classification":
        return RandomForestClassifier(
            n_estimators=100,
            random_state=random_state,
            n_jobs=-1,
        )

    return RandomForestRegressor(
        n_estimators=100,
        random_state=random_state,
        n_jobs=-1,
    )


def train_baseline(args):
    """
    Train baseline model and save artifact.
    """
    Path(args.model_dir).mkdir(parents=True, exist_ok=True)
    Path(args.report_dir).mkdir(parents=True, exist_ok=True)

    X_train, y_train, X_test, y_test = load_split_data(
        train_path=args.train_path,
        test_path=args.test_path,
        target_column=args.target_column,
    )

    problem_type = detect_problem_type(y_train)

    preprocessor, numeric_features, categorical_features = build_preprocessor(X_train)
    model = build_baseline_model(problem_type, args.random_state)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", model),
        ]
    )

    pipeline.fit(X_train, y_train)

    metrics = evaluate_model(
        model=pipeline,
        X_test=X_test,
        y_test=y_test,
        output_dir=args.report_dir,
    )

    model_path = Path(args.model_dir) / "baseline_model.pkl"
    joblib.dump(pipeline, model_path)

    metadata = {
        "target_column": args.target_column,
        "problem_type": problem_type,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "input_columns": X_train.columns.tolist(),
        "model_path": str(model_path),
        "metrics": metrics,
    }

    metadata_path = Path(args.model_dir) / "baseline_metadata.json"

    with open(metadata_path, "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=4)

    print("Baseline training completed.")
    print(f"Model saved to: {model_path}")
    print(f"Metadata saved to: {metadata_path}")
    print(metrics)


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--train-path", default="data/splits/train.csv")
    parser.add_argument("--test-path", default="data/splits/test.csv")
    parser.add_argument("--target-column", default="target")
    parser.add_argument("--model-dir", default="models")
    parser.add_argument("--report-dir", default="reports")
    parser.add_argument("--random-state", type=int, default=42)

    return parser.parse_args()


if __name__ == "__main__":
    train_baseline(parse_args())

Overwriting src/training/train_baseline.py


### **Temporary dummy data for testing only**

In [58]:
# =========================
# Temporary test data only
# Delete once real data exists
# =========================

import pandas as pd
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

Path("data/splits").mkdir(parents=True, exist_ok=True)

X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_classes=2,
    random_state=42,
)

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(8)])
df["target"] = y

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["target"],
)

train_df.to_csv("data/splits/train.csv", index=False)
test_df.to_csv("data/splits/test.csv", index=False)

print("Temporary train/test data created.")

Temporary train/test data created.


In [59]:
# =========================
# Run baseline model
# =========================

!python src/training/train.py \
    --train-path data/splits/train.csv \
    --test-path data/splits/test.csv \
    --target-column target

python3: can't open file '/content/src/training/train.py': [Errno 2] No such file or directory


## **Issue #20 — Log experiments**

### **Notebook Cell**

In [60]:
# =========================
# Issue #20: Log experiments
# File: src/training/train_mlflow.py
# =========================

In [61]:
%%writefile src/training/train_mlflow.py
# =========================
# Training with MLflow experiment logging
# =========================

import argparse
import json
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.evaluation.evaluate import detect_problem_type, evaluate_model
from src.training.mlflow_setup import configure_mlflow


def load_split_data(train_path, test_path, target_column):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    return X_train, y_train, X_test, y_test


def build_preprocessor(X_train):
    numeric_features = X_train.select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

    return preprocessor, numeric_features, categorical_features


def get_candidate_models(problem_type, random_state):
    if problem_type == "classification":
        return {
            "logistic_regression": LogisticRegression(max_iter=1000),
            "random_forest": RandomForestClassifier(
                n_estimators=200,
                random_state=random_state,
                n_jobs=-1,
            ),
            "gradient_boosting": GradientBoostingClassifier(random_state=random_state),
        }

    return {
        "ridge_regression": Ridge(),
        "random_forest": RandomForestRegressor(
            n_estimators=200,
            random_state=random_state,
            n_jobs=-1,
        ),
        "gradient_boosting": GradientBoostingRegressor(random_state=random_state),
    }


def get_primary_metric(problem_type):
    if problem_type == "classification":
        return "f1_macro", True

    return "rmse", False


def train_with_mlflow(args):
    Path(args.model_dir).mkdir(parents=True, exist_ok=True)
    Path(args.report_dir).mkdir(parents=True, exist_ok=True)

    configure_mlflow(
        experiment_name=args.experiment_name,
        backend_store_path=args.backend_store_path,
    )

    X_train, y_train, X_test, y_test = load_split_data(
        train_path=args.train_path,
        test_path=args.test_path,
        target_column=args.target_column,
    )

    problem_type = detect_problem_type(y_train)
    primary_metric, higher_is_better = get_primary_metric(problem_type)

    preprocessor, numeric_features, categorical_features = build_preprocessor(X_train)
    candidate_models = get_candidate_models(problem_type, args.random_state)

    best_score = None
    best_model_name = None
    best_model = None
    best_metrics = None

    for model_name, estimator in candidate_models.items():
        with mlflow.start_run(run_name=f"experiment_{model_name}"):
            pipeline = Pipeline(
                steps=[
                    ("preprocessor", preprocessor),
                    ("model", estimator),
                ]
            )

            pipeline.fit(X_train, y_train)

            metrics = evaluate_model(
                model=pipeline,
                X_test=X_test,
                y_test=y_test,
                output_dir=args.report_dir,
            )

            score = metrics[primary_metric]

            mlflow.log_param("model_name", model_name)
            mlflow.log_param("problem_type", problem_type)
            mlflow.log_param("target_column", args.target_column)
            mlflow.log_param("train_rows", X_train.shape[0])
            mlflow.log_param("test_rows", X_test.shape[0])
            mlflow.log_param("numeric_features", len(numeric_features))
            mlflow.log_param("categorical_features", len(categorical_features))

            for metric_name, metric_value in metrics.items():
                if isinstance(metric_value, (int, float)):
                    mlflow.log_metric(metric_name, metric_value)

            metrics_path = Path(args.report_dir) / "metrics.json"
            report_path = Path(args.report_dir) / "classification_report.txt"
            confusion_path = Path(args.report_dir) / "confusion_matrix.csv"

            if metrics_path.exists():
                mlflow.log_artifact(str(metrics_path))

            if report_path.exists():
                mlflow.log_artifact(str(report_path))

            if confusion_path.exists():
                mlflow.log_artifact(str(confusion_path))

            mlflow.sklearn.log_model(
                sk_model=pipeline,
                artifact_path="model",
            )

            should_replace = (
                best_score is None
                or (higher_is_better and score > best_score)
                or (not higher_is_better and score < best_score)
            )

            if should_replace:
                best_score = score
                best_model_name = model_name
                best_model = pipeline
                best_metrics = metrics

    best_model_path = Path(args.model_dir) / "best_logged_model.pkl"
    joblib.dump(best_model, best_model_path)

    summary = {
        "best_model_name": best_model_name,
        "primary_metric": primary_metric,
        "best_score": float(best_score),
        "metrics": best_metrics,
        "model_path": str(best_model_path),
    }

    summary_path = Path(args.report_dir) / "best_logged_model_summary.json"

    with open(summary_path, "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=4)

    print("MLflow experiment logging completed.")
    print(summary)


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--train-path", default="data/splits/train.csv")
    parser.add_argument("--test-path", default="data/splits/test.csv")
    parser.add_argument("--target-column", default="target")
    parser.add_argument("--model-dir", default="models")
    parser.add_argument("--report-dir", default="reports")
    parser.add_argument("--experiment-name", default="mlops_training_experiments")
    parser.add_argument("--backend-store-path", default="mlruns/mlflow.db")
    parser.add_argument("--random-state", type=int, default=42)

    return parser.parse_args()


if __name__ == "__main__":
    train_with_mlflow(parse_args())

Overwriting src/training/train_mlflow.py


In [62]:
# =========================
# Run MLflow experiments
# =========================

!python src/training/train.py \
    --train-path data/splits/train.csv \
    --test-path data/splits/test.csv \
    --target-column target

python3: can't open file '/content/src/training/train.py': [Errno 2] No such file or directory


## **Issue #21 — Hyperparameter tuning**

### **Notebook Cell**

In [63]:
# =========================
# Issue #21: Hyperparameter tuning
# File: src/training/hpo.py
# =========================

In [64]:
%%writefile src/training/hpo.py
# =========================
# Hyperparameter tuning with MLflow
# =========================

import argparse
import json
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src.evaluation.evaluate import detect_problem_type, evaluate_model
from src.training.mlflow_setup import configure_mlflow


def load_split_data(train_path, test_path, target_column):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    X_test = test_df.drop(columns=[target_column])
    y_test = test_df[target_column]

    return X_train, y_train, X_test, y_test


def build_preprocessor(X_train):
    numeric_features = X_train.select_dtypes(
        include=["int64", "float64", "int32", "float32"]
    ).columns.tolist()

    categorical_features = X_train.select_dtypes(
        include=["object", "category", "bool"]
    ).columns.tolist()

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ]
    )

    return preprocessor


def build_search(problem_type, random_state):
    if problem_type == "classification":
        estimator = RandomForestClassifier(random_state=random_state, n_jobs=-1)
        scoring = "f1_macro"
    else:
        estimator = RandomForestRegressor(random_state=random_state, n_jobs=-1)
        scoring = "neg_root_mean_squared_error"

    param_distributions = {
        "model__n_estimators": [100, 200, 300, 500],
        "model__max_depth": [None, 5, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf": [1, 2, 4],
    }

    return estimator, param_distributions, scoring


def run_hpo(args):
    Path(args.model_dir).mkdir(parents=True, exist_ok=True)
    Path(args.report_dir).mkdir(parents=True, exist_ok=True)

    configure_mlflow(
        experiment_name=args.experiment_name,
        backend_store_path=args.backend_store_path,
    )

    X_train, y_train, X_test, y_test = load_split_data(
        train_path=args.train_path,
        test_path=args.test_path,
        target_column=args.target_column,
    )

    problem_type = detect_problem_type(y_train)

    preprocessor = build_preprocessor(X_train)
    estimator, param_distributions, scoring = build_search(problem_type, args.random_state)

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=args.n_iter,
        scoring=scoring,
        cv=args.cv,
        n_jobs=-1,
        random_state=args.random_state,
        verbose=1,
    )

    with mlflow.start_run(run_name="random_forest_hpo"):
        search.fit(X_train, y_train)

        best_model = search.best_estimator_

        metrics = evaluate_model(
            model=best_model,
            X_test=X_test,
            y_test=y_test,
            output_dir=args.report_dir,
        )

        model_path = Path(args.model_dir) / "tuned_model.pkl"
        joblib.dump(best_model, model_path)

        summary = {
            "problem_type": problem_type,
            "scoring": scoring,
            "best_cv_score": float(search.best_score_),
            "best_params": search.best_params_,
            "test_metrics": metrics,
            "model_path": str(model_path),
        }

        summary_path = Path(args.report_dir) / "hpo_summary.json"

        with open(summary_path, "w", encoding="utf-8") as file:
            json.dump(summary, file, indent=4)

        mlflow.log_param("model_name", "random_forest_hpo")
        mlflow.log_param("problem_type", problem_type)
        mlflow.log_param("scoring", scoring)
        mlflow.log_param("cv", args.cv)
        mlflow.log_param("n_iter", args.n_iter)

        for param_name, param_value in search.best_params_.items():
            mlflow.log_param(param_name, param_value)

        mlflow.log_metric("best_cv_score", float(search.best_score_))

        for metric_name, metric_value in metrics.items():
            if isinstance(metric_value, (int, float)):
                mlflow.log_metric(metric_name, metric_value)

        mlflow.log_artifact(str(summary_path))
        mlflow.log_artifact(str(model_path))

        mlflow.sklearn.log_model(
            sk_model=best_model,
            artifact_path="model",
        )

    print("Hyperparameter tuning completed.")
    print(summary)


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--train-path", default="data/splits/train.csv")
    parser.add_argument("--test-path", default="data/splits/test.csv")
    parser.add_argument("--target-column", default="target")
    parser.add_argument("--model-dir", default="models")
    parser.add_argument("--report-dir", default="reports")
    parser.add_argument("--experiment-name", default="mlops_training_experiments")
    parser.add_argument("--backend-store-path", default="mlruns/mlflow.db")
    parser.add_argument("--random-state", type=int, default=42)
    parser.add_argument("--cv", type=int, default=3)
    parser.add_argument("--n-iter", type=int, default=10)

    return parser.parse_args()


if __name__ == "__main__":
    run_hpo(parse_args())

Overwriting src/training/hpo.py


In [65]:
# =========================
# Run hyperparameter tuning
# =========================

!python src/training/hpo.py \
    --train-path data/splits/train.csv \
    --test-path data/splits/test.csv \
    --target-column target \
    --n-iter 10 \
    --cv 3

Traceback (most recent call last):
  File "/content/src/training/hpo.py", line 20, in <module>
    from src.evaluation.evaluate import detect_problem_type, evaluate_model
ModuleNotFoundError: No module named 'src'


## **Issue #22 — Register best model**

### **Notebook Cell**

In [66]:
# =========================
# Issue #22: Register best model
# File: src/training/register_model.py
# =========================

In [67]:
%%writefile src/training/register_model.py
# =========================
# Register best model in MLflow
# =========================

import argparse
from pathlib import Path

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from src.training.mlflow_setup import configure_mlflow


def find_best_run(experiment_name, metric_name, higher_is_better=True):
    """
    Find the best MLflow run based on the selected metric.
    """
    client = MlflowClient()

    experiment = client.get_experiment_by_name(experiment_name)

    if experiment is None:
        raise ValueError(f"Experiment '{experiment_name}' does not exist.")

    order_by = [f"metrics.{metric_name} {'DESC' if higher_is_better else 'ASC'}"]

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=order_by,
        max_results=1,
    )

    if not runs:
        raise ValueError("No MLflow runs found.")

    return runs[0]


def register_and_promote_model(args):
    """
    Register best model and promote it to production.
    """
    configure_mlflow(
        experiment_name=args.experiment_name,
        backend_store_path=args.backend_store_path,
    )

    best_run = find_best_run(
        experiment_name=args.experiment_name,
        metric_name=args.metric_name,
        higher_is_better=args.higher_is_better,
    )

    run_id = best_run.info.run_id
    model_uri = f"runs:/{run_id}/model"

    result = mlflow.register_model(
        model_uri=model_uri,
        name=args.registered_model_name,
    )

    client = MlflowClient()

    client.transition_model_version_stage(
        name=args.registered_model_name,
        version=result.version,
        stage="Staging",
        archive_existing_versions=True,
    )

    client.transition_model_version_stage(
        name=args.registered_model_name,
        version=result.version,
        stage="Production",
        archive_existing_versions=True,
    )

    print("Best model registered and promoted.")
    print(f"Run ID: {run_id}")
    print(f"Model name: {args.registered_model_name}")
    print(f"Model version: {result.version}")
    print("Stage: Production")


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument("--experiment-name", default="mlops_training_experiments")
    parser.add_argument("--backend-store-path", default="mlruns/mlflow.db")
    parser.add_argument("--registered-model-name", default="BestMLOpsModel")
    parser.add_argument("--metric-name", default="f1_macro")
    parser.add_argument("--higher-is-better", action="store_true")

    return parser.parse_args()


if __name__ == "__main__":
    register_and_promote_model(parse_args())

Overwriting src/training/register_model.py


In [68]:
# =========================
# Register best classification model
# =========================

!python src/training/register_model.py \
    --experiment-name mlops_training_experiments \
    --registered-model-name BestMLOpsModel \
    --metric-name f1_macro \
    --higher-is-better

Traceback (most recent call last):
  File "/content/src/training/register_model.py", line 12, in <module>
    from src.training.mlflow_setup import configure_mlflow
ModuleNotFoundError: No module named 'src'


In [69]:
# =========================
# Register best regression model
# =========================

!python src/training/register_model.py \
    --experiment-name mlops_training_experiments \
    --registered-model-name BestMLOpsModel \
    --metric-name rmse

Traceback (most recent call last):
  File "/content/src/training/register_model.py", line 12, in <module>
    from src.training.mlflow_setup import configure_mlflow
ModuleNotFoundError: No module named 'src'


## **Issue #16 — DVC pipeline stages**

### **Notebook Cell**

In [70]:
# =========================
# Issue #16: DVC Pipeline Stages
# File: src/training/prepare_placeholder_data.py
# =========================

In [71]:
%%writefile src/training/prepare_placeholder_data.py
# =========================
# Placeholder data preparation
# =========================

from pathlib import Path

import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split


def create_placeholder_data(
    train_path="data/splits/train.csv",
    test_path="data/splits/test.csv",
    target_column="target",
):
    """
    Create placeholder train/test splits only if real split files do not exist.
    This prevents the DVC pipeline from failing before the real dataset is ready.
    """
    train_path = Path(train_path)
    test_path = Path(test_path)

    if train_path.exists() and test_path.exists():
        print("Train/test split files already exist. Placeholder data was not created.")
        return

    train_path.parent.mkdir(parents=True, exist_ok=True)

    X, y = make_classification(
        n_samples=500,
        n_features=8,
        n_informative=5,
        n_classes=2,
        random_state=42,
    )

    df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(8)])
    df[target_column] = y

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df[target_column],
    )

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Placeholder train/test data created.")
    print(f"Train path: {train_path}")
    print(f"Test path: {test_path}")


if __name__ == "__main__":
    create_placeholder_data()

Overwriting src/training/prepare_placeholder_data.py


**dvc.yaml**

In [72]:
# =========================
# Create dvc.yaml
# =========================

In [73]:
%%writefile dvc.yaml
stages:
  prepare:
    cmd: python -c "from pathlib import Path; Path('data/splits').mkdir(parents=True, exist_ok=True); print('Prepare stage completed')"
    outs:
      - data/splits

  preprocess:
    cmd: python -c "print('Preprocess stage uses outputs from data/preprocessing lead')"
    deps:
      - data/splits/train.csv
      - data/splits/test.csv

  featurize:
    cmd: python -c "print('Featurize stage uses processed split files')"
    deps:
      - data/splits/train.csv
      - data/splits/test.csv

  train:
    cmd: python src/training/train_mlflow.py --train-path data/splits/train.csv --test-path data/splits/test.csv --target-column target
    deps:
      - src/training/train_mlflow.py
      - src/training/mlflow_setup.py
      - src/evaluation/evaluate.py
      - data/splits/train.csv
      - data/splits/test.csv
    outs:
      - models/best_logged_model.pkl
    metrics:
      - reports/metrics.json:
          cache: false

Overwriting dvc.yaml


In [74]:
# =========================
# Initialize DVC safely
# =========================

!if [ ! -d ".dvc" ]; then dvc init; else echo "DVC is already initialized."; fi

ERROR: failed to initiate DVC - /content is not tracked by any supported SCM tool (e.g. Git). Use `--no-scm` if you don't want to use any SCM or `--subdir` if initializing inside a subdirectory of a parent SCM repository.


In [75]:
# =========================
# Run DVC pipeline
# =========================

!dvc repro

ERROR: you are not inside of a DVC repository (checked up to mount point '/')


In [76]:
# =========================
# Show DVC metrics
# =========================

!dvc metrics show

DVC failed to load some metrics for following revisions: ''.
Path
reports/metrics.json


## **Download**

In [77]:
# =========================
# Zip all generated project files/folders
# =========================

import shutil
from pathlib import Path

zip_name = "mlops_ibrahim_all_files"
output_zip = f"/content/{zip_name}.zip"

items_to_zip = [
    "data",
    "mlruns",
    "models",
    "reports",
    "sample_data",
    "src",
    "dvc.yaml",
    "mlflow.log",
]

temp_dir = Path("/content/mlops_ibrahim_export")
temp_dir.mkdir(parents=True, exist_ok=True)

# Copy selected files/folders into a temporary export folder
for item in items_to_zip:
    source = Path(item)

    if source.exists():
        destination = temp_dir / source.name

        if source.is_dir():
            if destination.exists():
                shutil.rmtree(destination)
            shutil.copytree(source, destination)
        else:
            shutil.copy2(source, destination)
    else:
        print(f"Skipped missing item: {item}")

# Create ZIP file
shutil.make_archive(
    base_name=f"/content/{zip_name}",
    format="zip",
    root_dir=temp_dir
)

print(f"ZIP created successfully: {output_zip}")

ZIP created successfully: /content/mlops_ibrahim_all_files.zip


In [ ]:
from google.colab import files

files.download("/content/mlops_ibrahim_all_files.zip")